# Fuente 1 — API del servidor Docker (pobreza multidimensional, IPM)
## ETL Segunda Entrega · Notebook de la fuente "API"

La API corre en un servidor Docker y entrega el archivo de hogares del IPM (pobreza multidimensional). Cada fila es un hogar con sus 15 indicadores de privación, el `ipm`, la marca `pobre` y los factores de expansión.

El notebook:
1. Extrae los datos desde la API.
2. Hace el EDA (estructura, nulos, duplicados, atípicos y consistencia interna).
3. Transforma y normaliza los datos.
4. Hace el EDA visual sobre los datos transformados.
5. Valida con Great Expectations 1.x y guarda los resultados y el registro de validación.

Las funciones `read_api`, `transform_api` y `validar` quedan listas para pasarse a las tareas del DAG de Airflow.

**Dirección de la API:** se busca primero en la variable de entorno `API_URL`, luego en los *Secretos* de Colab (icono de llave, panel izquierdo) y si no existe se usa `http://201.32.130.152/api/productos`.

## Paso 0 — Dependencias

In [ ]:
!pip install -q "great-expectations>=1.0,<2" pandas pyarrow requests matplotlib

## Paso 1 — Configuración

In [ ]:
import os, io, json, getpass, warnings, contextlib
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/My Drive/Colab Notebooks/etl_proyecto"
except Exception:
    BASE = "./etl_proyecto"

INTERMEDIOS = f"{BASE}/staging"
REPORTES = f"{BASE}/reports"
for carpeta in (INTERMEDIOS, REPORTES):
    os.makedirs(carpeta, exist_ok=True)

In [ ]:
def pedir(nombre, mensaje, oculto=False, defecto=None):
    # orden de búsqueda: variable de entorno, secretos de colab, valor por defecto, teclado
    if nombre in os.environ:
        return os.environ[nombre]
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        pass
    if defecto is not None:
        return defecto
    return getpass.getpass(mensaje) if oculto else input(mensaje)


API_URL = pedir("API_URL", "", defecto="http://201.32.130.152/api/productos")
TIMEOUT = 120
print("API:", API_URL)
print("Salidas en:", BASE)

### Constantes del IPM
Los 15 indicadores valen 0 o 1 (1 = el hogar está privado). El `ipm` es la suma de cada indicador multiplicado por su peso, y un hogar es pobre (`pobre = 1`) cuando el `ipm` es mayor o igual a 1/3.

In [ ]:
PESOS_IPM = {
    # condiciones educativas del hogar (0.2)
    "logro_educativo": 0.10, "analfabetismo": 0.10,
    # niñez y juventud (0.2)
    "inasistencia_escolar": 0.05, "rezago_escolar": 0.05,
    "atencion_integral": 0.05, "trabajo_infantil": 0.05,
    # trabajo (0.2)
    "desempleo_larga_duracion": 0.10, "empleo_formal": 0.10,
    # salud (0.2)
    "aseguramiento_salud": 0.10, "barreras_acceso_salud": 0.10,
    # vivienda y servicios públicos (0.2)
    "acueducto": 0.04, "alcantarillado": 0.04, "pisos": 0.04,
    "paredes": 0.04, "hacinamiento": 0.04,
}
assert abs(sum(PESOS_IPM.values()) - 1) < 1e-9

DEPARTAMENTOS = {
    "05": "Antioquia", "08": "Atlántico", "11": "Bogotá D.C.", "13": "Bolívar",
    "15": "Boyacá", "17": "Caldas", "18": "Caquetá", "19": "Cauca", "20": "Cesar",
    "23": "Córdoba", "25": "Cundinamarca", "27": "Chocó", "41": "Huila",
    "44": "La Guajira", "47": "Magdalena", "50": "Meta", "52": "Nariño",
    "54": "Norte de Santander", "63": "Quindío", "66": "Risaralda",
    "68": "Santander", "70": "Sucre", "73": "Tolima", "76": "Valle del Cauca",
    "81": "Arauca", "85": "Casanare", "86": "Putumayo",
    "88": "San Andrés y Providencia", "91": "Amazonas", "94": "Guainía",
    "95": "Guaviare", "97": "Vaupés", "99": "Vichada",
}

# columnas casi vacías que no se usan en el análisis
COLS_DESCARTAR = ["p1075", "p1077s21", "p1077s22", "p1077s23"]

COLS_NUM = (["directorio", "secuencia_p", "secuencia_encuesta", "p5010", "p8526", "p8530",
             "region", "departamento", "personas", "fex_c", "fexp", "ipm", "pobre", "periodo"]
            + list(PESOS_IPM) + COLS_DESCARTAR)

## Paso 2 — Extracción (`read_api`)

La función pide los datos con reintentos y espera creciente entre intentos, y con un tiempo máximo de 120 segundos por petición. Si la respuesta es una lista, la usa directamente; si es un objeto, busca la lista de registros dentro de él (`data`, `results`, `items`, etc.). Si el servidor entrega la respuesta por páginas con un enlace `next`, lo sigue.

La primera petición imprime el tipo de respuesta y sus llaves, para confirmar el formato real de la API.

In [ ]:
def _sesion():
    s = requests.Session()
    retry = Retry(total=4, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
    s.mount("http://", HTTPAdapter(max_retries=retry))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def _registros(payload):
    """Devuelve (lista_de_registros, url_siguiente_o_None)."""
    if isinstance(payload, list):
        return payload, None
    if isinstance(payload, dict):
        siguiente = payload.get("next") or payload.get("next_page")
        siguiente = siguiente if isinstance(siguiente, str) and siguiente.startswith("http") else None
        for k in ("data", "results", "items", "productos", "rows", "records"):
            if isinstance(payload.get(k), list):
                return payload[k], siguiente
        for v in payload.values():
            if isinstance(v, list):
                return v, siguiente
    raise ValueError(f"No se encontró la lista de registros en la respuesta: {str(payload)[:300]}")


def read_api(url=API_URL, max_paginas=1000):
    s = _sesion()
    registros, paginas = [], 0
    while url and paginas < max_paginas:
        r = s.get(url, timeout=TIMEOUT)
        r.raise_for_status()
        payload = r.json()
        if paginas == 0:
            forma = list(payload)[:10] if isinstance(payload, dict) else f"lista de {len(payload):,} elementos"
            print(f"[read_api] código {r.status_code} | tipo: {type(payload).__name__} | contenido: {forma}")
        lote, url = _registros(payload)
        registros.extend(lote)
        paginas += 1
    df = pd.DataFrame(registros)
    print(f"[read_api] {len(df):,} filas x {df.shape[1]} columnas en {paginas} petición(es)")
    return df

In [ ]:
df_raw = read_api()
df_raw.to_csv(f"{INTERMEDIOS}/ipm_api_raw.csv", index=False)
df_raw.head()

## Paso 3 — EDA sobre los datos crudos

Para las estadísticas se usa una copia con nombres en minúscula, el texto `"NA"` convertido a nulo y las columnas numéricas convertidas a número. La transformación del Paso 5 usa esta misma función.

In [ ]:
def normalizar(df):
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]
    d = d.replace({"NA": np.nan, "": np.nan})
    for c in COLS_NUM:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    return d


tmp = normalizar(df_raw)
tmp.shape

In [ ]:
df_raw.dtypes.to_frame("tipo")

In [ ]:
nulos = df_raw.isnull().sum()
(nulos[nulos > 0].to_frame("nulos").assign(porcentaje=lambda t: (t["nulos"] / len(df_raw) * 100).round(1))
 if (nulos > 0).any() else "Sin nulos reales")

In [ ]:
# el texto "NA" cuenta como dato presente hasta que se convierte a nulo
texto_na = pd.Series({c: int((df_raw[c].astype(str).str.upper() == "NA").sum()) for c in df_raw.columns})
texto_na = texto_na[texto_na > 0]
(texto_na.to_frame("filas_con_NA_como_texto").assign(porcentaje=lambda t: (t["filas_con_NA_como_texto"] / len(df_raw) * 100).round(1))
 if len(texto_na) else "Sin valores NA escritos como texto")

In [ ]:
print("Duplicados exactos:", df_raw.duplicated().sum())

In [ ]:
LLAVE = ["directorio", "secuencia_p"]
print(f"Duplicados por llave {LLAVE}:", tmp.duplicated(subset=LLAVE).sum())

In [ ]:
cols_num = [c for c in ("personas", "fex_c", "fexp", "ipm") if c in tmp.columns]
tmp[cols_num].describe().T.round(2)

In [ ]:
# atípicos por rango intercuartílico
filas = []
for c in cols_num:
    q1, q3 = tmp[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = int(((tmp[c] < q1 - 1.5 * iqr) | (tmp[c] > q3 + 1.5 * iqr)).sum())
    filas.append({"variable": c, "atipicos": n, "porcentaje": round(n / tmp[c].notna().sum() * 100, 1)})
atipicos = pd.DataFrame(filas)
atipicos

### Consistencia interna del dataset
Antes de transformar se comprueba que los datos cumplen la lógica del IPM:
- `ipm` debe ser la suma ponderada de los 15 indicadores.
- `pobre` debe ser 1 cuando el `ipm` es mayor o igual a 1/3.
- `fexp` debe ser `fex_c × personas` (factor de expansión de personas).

In [ ]:
def chequeos_consistencia(d):
    ind = [c for c in PESOS_IPM if c in d.columns]
    ipm_rec = sum(d[c] * PESOS_IPM[c] for c in ind)
    return pd.DataFrame({
        "ipm_coincide": (d["ipm"] - ipm_rec).abs() < 0.005,
        "pobre_coincide": d["pobre"] == (ipm_rec >= 1 / 3 - 1e-9).astype(int),
        "fexp_coincide": np.isclose(d["fexp"], d["fex_c"] * d["personas"], rtol=1e-4),
    })


res = chequeos_consistencia(tmp)
(res.mean() * 100).round(2).to_frame("% de filas consistentes")

In [ ]:
if "secuencia_encuesta" in tmp.columns:
    print("secuencia_encuesta igual a personas en todas las filas:", bool((tmp["secuencia_encuesta"] == tmp["personas"]).all()))
else:
    print("La columna secuencia_encuesta no existe")

In [ ]:
if "periodo" in tmp.columns:
    print(f"Nulos en periodo: {tmp['periodo'].isna().mean():.1%}")

In [ ]:
if "periodo" in tmp.columns:
    print("Valores distintos de periodo:", sorted(tmp["periodo"].dropna().unique())[:10])

## Paso 4 — Primer vistazo a los departamentos
Se revisa que los códigos de departamento que llegan de la API existan en el catálogo antes de transformar.

In [ ]:
codigos = tmp["departamento"].dropna().astype(int).astype(str).str.zfill(2)
fuera = sorted(set(codigos) - set(DEPARTAMENTOS))
print("Códigos fuera del catálogo:", fuera if fuera else "ninguno")

## Paso 5 — Transformación (`transform_api`)

Decisiones (quedan documentadas para el informe):
- Nombres de columnas en minúscula y `"NA"` convertido a nulo real.
- `departamento` pasa de entero (`5`, `8`) a texto de 2 dígitos (`"05"`, `"08"`) en `codigo_departamento` y se agrega `nombre_departamento`. Es la llave para cruzar por geografía.
- Se crea `id_hogar = directorio-secuencia_p`, igual al que arman las otras fuentes, para poder unirlas.
- `periodo` (entero `aaaammdd`) solo viene lleno en una parte de las filas. Se convierte a fecha y se crea `fecha_corte` con la fecha más frecuente, para todas las filas.
- Se agrega `n_privaciones` (cuántos de los 15 indicadores valen 1).
- Se descartan `p1075` y `p1077s21` a `p1077s23` porque casi todas sus filas son nulas (lista en `COLS_DESCARTAR`).
- `secuencia_encuesta` se descarta solo si es idéntica a `personas`.
- Se eliminan filas completamente duplicadas.

In [ ]:
def transform_api(df_raw):
    df = normalizar(df_raw)

    antes = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"[transform_api] duplicados exactos eliminados: {antes - len(df):,}")

    # departamento: código de 2 dígitos y nombre
    dep = df["departamento"].astype("Int64")
    df["codigo_departamento"] = dep.astype("string").str.zfill(2)
    df["nombre_departamento"] = df["codigo_departamento"].map(DEPARTAMENTOS)
    sin_nombre = int(df["nombre_departamento"].isna().sum())
    if sin_nombre:
        print(f"[transform_api] ATENCIÓN: {sin_nombre:,} filas con departamento fuera del catálogo")

    # llave del hogar, igual a la de las otras fuentes
    df["id_hogar"] = (df["directorio"].astype("Int64").astype("string") + "-"
                      + df["secuencia_p"].astype("Int64").astype("string"))

    # fecha de corte: el periodo viene lleno solo en parte de las filas
    if "periodo" in df.columns:
        texto = df["periodo"].astype("Int64").astype("string")
        fechas = pd.to_datetime(texto, format="%Y%m%d", errors="coerce")
        moda = fechas.dropna().mode()
        df["fecha_corte"] = moda.iloc[0] if len(moda) else pd.NaT
        print("[transform_api] fecha_corte:", df["fecha_corte"].iloc[0])
        df = df.drop(columns=["periodo"])

    # privaciones
    ind = [c for c in PESOS_IPM if c in df.columns]
    df["n_privaciones"] = df[ind].sum(axis=1).astype(int)

    # columnas descartadas
    df = df.drop(columns=[c for c in COLS_DESCARTAR if c in df.columns])
    if "secuencia_encuesta" in df.columns and (df["secuencia_encuesta"] == df["personas"]).all():
        df = df.drop(columns=["secuencia_encuesta"])

    print(f"[transform_api] resultado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    return df


df = transform_api(df_raw)
df.head()

In [ ]:
df.dtypes.to_frame("tipo")

In [ ]:
df["nombre_departamento"].value_counts(dropna=False).to_frame("hogares")

In [ ]:
colu_api = ['departamento', 'ipm', 'fex_c', 'nombre_departamento']
api_ipm = df[colu_api].copy()

In [ ]:
api_ipm.head(20)

In [ ]:
api_ipm['ponderado'] = api_ipm['ipm'] * api_ipm['fex_c']

In [ ]:
rsumido = api_ipm.groupby('nombre_departamento').agg(
    hogares=('ipm', 'size'),
    ipm_promedio=('ipm', 'mean'),
    suma_ponderado=('ponderado', 'sum'),
    suma_fex_c=('fex_c', 'sum'),
)
rsumido['ipm_ponderado'] = rsumido['suma_ponderado'] / rsumido['suma_fex_c']
rsumido.head(20)

In [ ]:
ipm_rsmdo = rsumido['ipm_ponderado']

In [ ]:
ipm_rsmdo.sort_values(ascending=False)

In [ ]:
ipm_rsmdo

## Paso 6 — EDA visual (sobre los datos transformados)
Los porcentajes de **personas** usan `fexp`; los de **hogares** usan `fex_c`.

In [ ]:
def pct_pond(valor, peso):
    return (valor * peso).sum() / peso.sum() * 100

In [ ]:
print(f"Hogares pobres (ponderado por hogares, fex_c): {pct_pond(df['pobre'], df['fex_c']):.1f}%")

In [ ]:
print(f"Personas en pobreza multidimensional (ponderado por personas, fexp): {pct_pond(df['pobre'], df['fexp']):.1f}%")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["ipm"], bins=30, weights=df["fexp"], color="#2980b9")
plt.axvline(1 / 3, color="#c0392b", linestyle="--", label="Umbral de pobreza (1/3)")
plt.title("Distribución del IPM (ponderada por personas)")
plt.xlabel("IPM"); plt.ylabel("Personas (expandidas)"); plt.legend()
plt.savefig(f"{REPORTES}/eda_api_ipm.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
df["n_privaciones"].value_counts().sort_index().plot(kind="bar", color="#8e44ad")
plt.title("Hogares por número de privaciones (sin ponderar)")
plt.xlabel("Privaciones (de 15)"); plt.ylabel("Hogares en la muestra")
plt.savefig(f"{REPORTES}/eda_api_privaciones_hogar.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# tasa de privación por indicador (% de personas privadas)
ind = [c for c in PESOS_IPM if c in df.columns]
tasas = pd.Series({c: pct_pond(df[c], df["fexp"]) for c in ind}).sort_values()
tasas.round(1).to_frame("% de personas privadas")

In [ ]:
plt.figure(figsize=(8, 5))
tasas.plot(kind="barh", color="#e67e22")
plt.title("Privación por indicador (% de personas)")
plt.xlabel("%")
plt.savefig(f"{REPORTES}/eda_api_indicadores.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# pobreza multidimensional por departamento (personas)
tmp_dep = df.assign(_pobre=df["pobre"] * df["fexp"], _ipm=df["ipm"] * df["fexp"])
dep = (tmp_dep.groupby("nombre_departamento")
              .agg(pobres=("_pobre", "sum"), ipm_pond=("_ipm", "sum"),
                   personas=("fexp", "sum"), hogares_muestra=("id_hogar", "count")))
dep["pct_personas_pobres"] = dep["pobres"] / dep["personas"] * 100
dep["ipm_promedio"] = dep["ipm_pond"] / dep["personas"]
dep = dep[["pct_personas_pobres", "ipm_promedio", "hogares_muestra"]].sort_values("pct_personas_pobres", ascending=False)
dep.round(2)

In [ ]:
plt.figure(figsize=(8, 6))
dep["pct_personas_pobres"].sort_values().plot(kind="barh", color="#c0392b")
plt.title("% de personas en pobreza multidimensional por departamento")
plt.xlabel("%")
plt.savefig(f"{REPORTES}/eda_api_departamentos.png", dpi=120, bbox_inches="tight")
plt.show()

Los departamentos con pocos hogares en la muestra tienen estimaciones poco confiables; por eso la columna `hogares_muestra` acompaña a cada porcentaje.

## Paso 7 — Hallazgos del EDA (resumen automático)
Las cifras y las frases salen de las celdas anteriores y sirven como base para el informe. Para los departamentos solo se consideran los que tienen al menos 30 hogares en la muestra.

In [ ]:
hallazgos = []

# estructura y calidad
hallazgos.append(f"La API entregó {len(df_raw):,} hogares y {df_raw.shape[1]} columnas.")
nulos_ = tmp.isnull().sum()
nulos_ = nulos_[nulos_ > 0].sort_values(ascending=False)
if len(nulos_):
    top = ", ".join(f"{c} ({n / len(tmp):.1%})" for c, n in nulos_.head(5).items())
    hallazgos.append(f"Columnas con más nulos una vez convertido el texto NA: {top}. Las casi vacías se descartan en la transformación.")
hallazgos.append(f"Duplicados exactos: {df_raw.duplicated().sum():,}; duplicados por llave directorio + secuencia_p: {tmp.duplicated(subset=LLAVE).sum():,}.")

# consistencia
for nombre, valor in (res.mean() * 100).items():
    hallazgos.append(f"Consistencia {nombre}: {valor:.2f}% de las filas.")

# atípicos
con_atip = atipicos[atipicos["atipicos"] > 0]
if len(con_atip):
    detalle = ", ".join(f"{r.variable} ({r.porcentaje}%)" for r in con_atip.itertuples())
    hallazgos.append(f"Atípicos por rango intercuartílico: {detalle}. "
                     "Se conservan porque los factores de expansión varían por el diseño de la muestra.")

# patrones
hallazgos.append(f"Hogares pobres: {pct_pond(df['pobre'], df['fex_c']):.1f}%; personas en pobreza multidimensional: {pct_pond(df['pobre'], df['fexp']):.1f}%.")
hallazgos.append(f"La privación más frecuente es {tasas.idxmax()} ({tasas.max():.1f}% de las personas) y la menos frecuente es {tasas.idxmin()} ({tasas.min():.1f}%).")
dep_ok = dep[dep["hogares_muestra"] >= 30]
if len(dep_ok):
    hallazgos.append(f"Mayor pobreza multidimensional: {dep_ok['pct_personas_pobres'].idxmax()} ({dep_ok['pct_personas_pobres'].max():.1f}% de las personas). "
                     f"Menor: {dep_ok['pct_personas_pobres'].idxmin()} ({dep_ok['pct_personas_pobres'].min():.1f}%).")
hallazgos.append(f"Departamentos con menos de 30 hogares en la muestra: {int((dep['hogares_muestra'] < 30).sum())}.")

for linea in hallazgos:
    print("-", linea)

**Qué significan para la transformación**
- Si las comprobaciones de consistencia dan 100%, no hace falta corregir `ipm`, `pobre` ni `fexp`, y las reglas de validación pueden exigirlas sin tolerancia (o con una tolerancia muy pequeña).
- El texto `"NA"` se trata como nulo para que las columnas se lean como números y los nulos se cuenten bien.
- Los factores de expansión (`fex_c`, `fexp`) deben usarse como pesos en cualquier porcentaje, porque cada hogar representa un número distinto de hogares y personas en la población.
- El departamento queda como código de 2 dígitos y como nombre, y `id_hogar` queda como llave para el `merge` con las otras dos fuentes.
- Las estimaciones de los departamentos con pocos hogares en la muestra no deben compararse como si fueran igual de precisas que las de los demás.

In [ ]:
with open(f"{REPORTES}/hallazgos_api.md", "w", encoding="utf-8") as f:
    f.write("# Hallazgos del EDA — fuente API (IPM)\n\n" + "\n".join(f"- {h}" for h in hallazgos) + "\n")

## Paso 8 — Validación con Great Expectations 1.x
Cada validación usa una `ExpectationSuite` real. Cada regla se guarda en el registro `validation_log.csv` (una fila por regla, con el mismo formato que el notebook 03), además del resumen en JSON.

In [ ]:
import great_expectations as gx
import great_expectations.expectations as gxe

LOG_VALIDACION = f"{REPORTES}/validation_log.csv"


def validar(df, nombre_suite, expectativas, etiqueta="manual"):
    d = df.copy()
    for c in d.columns:   # great expectations trabaja mejor con tipos normales que con los que admiten nulos
        if str(d[c].dtype) in ("Int64", "string"):
            d[c] = d[c].astype("float64") if str(d[c].dtype) == "Int64" else d[c].astype(object)

    contexto = gx.get_context(mode="ephemeral")
    activo = contexto.data_sources.add_pandas("origen").add_dataframe_asset(nombre_suite)
    lote = activo.add_batch_definition_whole_dataframe("lote").get_batch(batch_parameters={"dataframe": d})

    suite = gx.ExpectationSuite(name=nombre_suite)
    for e in expectativas:
        suite.add_expectation(e)
    with contextlib.redirect_stderr(io.StringIO()):   # oculta las barras de progreso
        resultado = lote.validate(suite)

    filas = []
    for r in resultado.results:
        kw = {k: v for k, v in r.expectation_config.kwargs.items() if k != "batch_id"}
        filas.append({
            "fecha": datetime.now().isoformat(timespec="seconds"), "suite": nombre_suite, "run": etiqueta,
            "regla": r.expectation_config.type, "parametros": json.dumps(kw, ensure_ascii=False, default=str),
            "ok": bool(r.success),
            "porcentaje_fuera": r.result.get("unexpected_percent"),
        })
    log = pd.DataFrame(filas)
    log.to_csv(LOG_VALIDACION, mode="a", header=not os.path.exists(LOG_VALIDACION), index=False)

    fallidas = log[~log["ok"]]
    resumen = {"suite": nombre_suite, "run": etiqueta, "passed": bool(resultado.success),
               "reglas_total": len(log), "fallidas": fallidas[["regla", "parametros", "porcentaje_fuera"]].to_dict("records")}
    with open(f"{REPORTES}/ge_{nombre_suite}_{etiqueta}.json", "w", encoding="utf-8") as f:
        json.dump(resumen, f, indent=2, ensure_ascii=False, default=str)

    print(f"[GE] {nombre_suite} -> {'PASS' if resumen['passed'] else 'FAIL'} "
          f"({len(fallidas)} de {len(log)} reglas fallidas)")
    for _, f_ in fallidas.iterrows():
        print("   ✗", f_["regla"], f_["parametros"], "->", f_["porcentaje_fuera"], "% fuera")
    return resumen

### Suite de la fuente API
Reglas propias del IPM: indicadores en {0, 1}, `ipm` entre 0 y 1, `pobre` coherente con el umbral 1/3, `fexp` coherente con `fex_c × personas`, llave única y departamentos válidos. Las comprobaciones de consistencia se agregan como columnas `chk_...` (1 = cumple) solo para validar; no se guardan en el resultado.

In [ ]:
def preparar_validacion(df):
    chk = chequeos_consistencia(df).astype(int).add_prefix("chk_")
    return pd.concat([df, chk], axis=1)


def reglas_api(df):
    ind = [c for c in PESOS_IPM if c in df.columns]
    e = []
    for c in ["id_hogar", "directorio", "secuencia_p", "codigo_departamento", "nombre_departamento",
              "personas", "fex_c", "fexp", "ipm", "pobre", "fecha_corte"] + ind:
        e.append(gxe.ExpectColumnValuesToNotBeNull(column=c))
    for c in ind + ["pobre"]:
        e.append(gxe.ExpectColumnValuesToBeInSet(column=c, value_set=[0, 1]))
    e += [
        gxe.ExpectTableRowCountToBeBetween(min_value=1),
        gxe.ExpectColumnValuesToBeUnique(column="id_hogar"),
        gxe.ExpectCompoundColumnsToBeUnique(column_list=["directorio", "secuencia_p"]),
        gxe.ExpectColumnValuesToBeBetween(column="ipm", min_value=0, max_value=1),
        gxe.ExpectColumnValuesToBeBetween(column="personas", min_value=1, max_value=30),
        gxe.ExpectColumnValuesToBeBetween(column="n_privaciones", min_value=0, max_value=15),
        gxe.ExpectColumnValuesToBeBetween(column="fex_c", min_value=0, strict_min=True),
        gxe.ExpectColumnValuesToBeBetween(column="fexp", min_value=0, strict_min=True),
        gxe.ExpectColumnValuesToBeInSet(column="codigo_departamento", value_set=list(DEPARTAMENTOS)),
        gxe.ExpectColumnValuesToBeInSet(column="chk_pobre_coincide", value_set=[1]),
        gxe.ExpectColumnValuesToBeInSet(column="chk_ipm_coincide", value_set=[1], mostly=0.99),
        gxe.ExpectColumnValuesToBeInSet(column="chk_fexp_coincide", value_set=[1], mostly=0.99),
    ]
    return e


df_validar = preparar_validacion(df)
resumen = validar(df_validar, "suite_api_ipm", reglas_api(df_validar), etiqueta="notebook")

## Paso 9 — Guardar la salida para el `merge`
Solo se guarda como limpia si la validación pasa (igual que hará la compuerta del DAG). El nombre del archivo coincide con el que busca el notebook 03.

In [ ]:
if resumen["passed"]:
    df.to_parquet(f"{INTERMEDIOS}/ipm_api_clean.parquet", index=False)
    print(f"Guardado: {INTERMEDIOS}/ipm_api_clean.parquet ({len(df):,} filas)")
else:
    print("La validación falló: no se guardó el archivo limpio. Revisar las reglas fallidas arriba.")

### Siguiente paso
- Notebook 02: fuente **CSV** (`viviendas_pdtos.csv`).
- Notebook 03: fuente **base de datos** (ya disponible).
- Después: el notebook del DAG de Airflow con tres ramas de lectura y transformación, `merge` por hogar (`id_hogar`), validación, carga y visualización.

Para el DAG, copiar `read_api`, `normalizar`, `transform_api`, `chequeos_consistencia`, `preparar_validacion`, `reglas_api` y `validar` (con las constantes `PESOS_IPM`, `DEPARTAMENTOS`, `COLS_NUM` y `COLS_DESCARTAR`) a funciones de tarea.